# REINFORCE Tutorial
*Reinforcement Learning (2026), University of Milan*

*Lecturer: Matteo Papini*

In this tutorial we will:
1. Implement a custom **Linear Quadratic Regulator (LQR)** environment using [gymnasium](https://gymnasium.farama.org/index.html) and [numpy](https://numpy.org/)
2. Implement a **linear Gaussian policy** using [pytorch](https://docs.pytorch.org/tutorials/index.html)
3. Implement the **REINFORCE** algorithm

Follow-up exercises:

4. Implement the **variance reduction** techniques seen in class (baselines, GPOMDP)
5. Extend to **multi-dimensional actions**

In [ ]:
import gymnasium as gym
from gymnasium import spaces
from gymnasium.envs.registration import register, registry
import numpy as np
import warnings
import torch
import matplotlib.pyplot as plt

## 1. Linear Quadratic Regulator (LQR)

Consider a time-invariant dynamical system with **continuous states and actions**
$$
  \mathcal{S} = \mathbb{R}^{d_{\mathcal{S}}}, \qquad \mathcal{A} = \mathbb{R}^{d_{\mathcal{A}}}
$$
The initial state is sampled from a multivariate standard normal distribution
$$
  s_0 \sim \mathcal{N}(\mathbf{0}, I_{d_{\mathcal{S}}})
$$
where $I_d$ denotes the $d\times d$ identity matrix.

The next state is a **linear** function of the current state and the action plus some centered Gaussian noise:
$$
  s_{t+1} = As_t + Ba_t + \eta_t
$$
where
$$
A \in \mathbb{R}^{d_{\mathcal{S}}\times d_{\mathcal{S}}}, \qquad B \in \mathbb{R}^{d_{\mathcal{S}}\times d_{\mathcal{A}}}, \qquad \eta \sim \mathcal{N}(\mathbf{0}, \sigma_s I_{d_{\mathcal{S}}})
$$
and $\qquad 0<\sigma_s\ll 1$ is a small standard deviation.

Rewards are negative and **quadratic** in the state and the action (quadratic costs):
$$
  r(s_t,a_t) = -s_t^\top Q s_t - a_t^\top R a_t
$$
where
$$
  Q \in \mathbb{R}^{d_{\mathcal{S}}\times d_{\mathcal{S}}}, \qquad R \in \mathbb{R}^{d_{\mathcal{A}}\times d_{\mathcal{A}}}
$$
are symmetric positive definite matrices.

The goal is to bring the state to the origin as quickly as possible, but taking into account the cost of large actions. This system can be used to model or locally approximate many control problems, and is widely used in the optimal control literature.

We will consider a finite-horizon version with epsisodes of fixed length $T$, no discounting, and no terminal costs.

In particular, we will implement the following system with 2-dimensional states and scalar actions:
* $d_{\mathcal{S}} = 2$
* $d_{\mathcal{A}} = 1$
* $\sigma_s = 0.1$
* $A = \begin{bmatrix}
1 & \tfrac{1}{2} \\
\tfrac{1}{2} & 0
\end{bmatrix}$
* $B = [1,0.5]^\top$
* $Q = 0.5I_2$
* $R=0.25$
* $T=10$

We will implement it as a **gym environment** using the gymnasium library, starting from the following template

💡 If you want to try other LQRs, make sure the dimensions are coherent. Moreover, you should avoid *non-controllable* systems. A linear system is controllable is the following $d_{\mathcal{S}}\times d_{\mathcal{S}}d_{\mathcal{A}}$ matrix has rank $d_\mathcal{S}$:
$$
  C = [B, AB, A^2B, \dots A^{d_{\mathcal{S}}-1}B]
$$

The partial implementation of LQR below already includes a snippet of code that throws a warning if the system is non-controllable

In [ ]:
# Extend Env
class LQR(gym.Env):
    def __init__(self,
                 sigma = 0.1,
                 A = np.array([[1, 0.5],[0.5, 0]], dtype=float),
                 B = np.array([1, 0.5], dtype=float),
                 Q = 0.5 * np.eye(2, dtype=float),
                 R = 0.25
                 ):

        self.sigma = sigma
        self.A = A
        self.B = B
        self.Q = Q
        self.R = R

        self.ds = A.shape[0]
        self.da = 1

        # Controllability check
        powers = [self.B[..., None]]
        for _ in range(self.ds):
            powers.append(self.A @ powers[-1])
        C = np.concatenate(powers, axis=-1)
        if np.linalg.matrix_rank(C) < self.ds:
            warnings.warn("The system is not controllable!", UserWarning)

        # Observation (state) space
        ...

        # Action space
        ...

    def reset(self, seed=None, options=None):
        # Reset the state
        self.rng = np.random.default_rng(seed)

        ...

        return self.state, dict()

    def step(self, action, render=False):
        # Compute reward
        ...

        # Update state
        ...

        terminated = False
        truncated = False

        return self.state, reward.item(), terminated, truncated, dict()

    def render(self, mode='human', close=False):
        print(np.array2string(self.state))


# Register the new Env and set the episode length
env_id = "LQR-v0"
T = 10

if env_id in registry:
  del registry[env_id]
register(
  id=env_id,
  entry_point=LQR,
  max_episode_steps=T,
)

Instantiate the environment and check everything is ok

In [ ]:
env = gym.make("LQR-v0")

**Initial state**

Is the state of the desided shape?

Do the values look reasonable?

Do you get a different state each time you call the reset method?

In [ ]:
s_0, _ = env.reset()
s_0

**State transitions**

Does the state change?

Is the shape still correct? The values still reasonable?

Is the state different at each call of the step method?

In [ ]:
a_0 = 0.2
s_1, r_1, _, _, _ = env.step(a_0)
s_1

**Reward**

Is the reward a negative scalar?

In [ ]:
r_1

**Episode termination**

Does the step method return a true "truncated" flag after T steps?

In [ ]:
truncated = False
t = 0
env.reset()
while not truncated:
  _, _, _, truncated, _ = env.step(action=0)
  t += 1
t

## 2. Linear Gaussian Policy

Optimal control theory (not covered in this class) shows that **linear policies** (in the state) are optimal for LQR
$$
  \pi_{\mathbb{\theta}}(s) = \mathbb{\theta}^\top s
$$
These are deterministic policies. Since policy gradient methods work with stochastic policies, we add some Gaussian noise
$$
  \pi_{\mathbb{\theta}}(a|s) = \mathcal{N}(\mathbb{\theta}^\top s; \sigma_a^2)
$$
We will later use REINFORCE to learn the optimal $\mathbb{\theta}$. The standard deviation $\sigma_a > 0$ is a hyperparameter. We use $\sigma_a=0.1$ as a default value. We will use $\mathbb{\theta}_0=[0,0]^\top$ as initial parameters.

REINFORCE needs to differentiate the policy w.r.t. its parameters, so we implement the policy using **pytorch** to support automatic differentiation.
The proposed implementation of the policy is stateful (it memorizes policy parameters). We will use CPU.

In [ ]:
class LinearGaussianPolicy:

  def __init__(self,
               sigma=0.1,
               theta=torch.zeros(2).float(),
               seed=None):

    self.sigma = sigma
    self.theta = theta.requires_grad_(True)
    self.rng = np.random.default_rng(seed) # Random number generator

  def seed(self, seed):
    self.rng = np.random.default_rng(seed)

  def set_theta(self, theta):
    self.theta = theta.float().requires_grad_(True)

  # Mean action for given state
  def mu(self, s):
    return ...

  # Logarithm of (unnormalized) probability density function
  # Implement to process batches of states and actions
  def log_pdf(self, states, actions):
    log_pdf = ...
    return log_pdf

  # Action sampling
  def act(self, s):
      return ...

Instantiate the policy and check everything is ok

**Sampling actions**

Is the action close to zero? (With zero parameters, it should be just noise)

Is it different at every call of the act method?

Do you get both positive and negative actions?

In [ ]:
pi = LinearGaussianPolicy()
env = gym.make("LQR-v0")
s, _ = env.reset()
a = pi.act(s)
a

Change the parameters: do actions change accordingly?

In [ ]:
pi.set_theta(10 * torch.ones(2))
a_10 = pi.act(s)
a_10

Try different states: do actions change accordingly?

In [ ]:
s_10 = 10 * s
a_100 = pi.act(s_10)
a_100

**Computing scores**

Is the log pdf differentiable?

In [ ]:
log_pdf = pi.log_pdf(torch.from_numpy(s).float(), a)
log_pdf

## 3. REINFORCE

To implement the REINFORCE algorithm, we first need a method to simulate episodes and collect trajectories.

The following implementation collects a batch of trajectories as three separate tensors: states, actions, and rewards.

We will use a default batch size of 100 trajectories.

In [ ]:
def collect_batch(env,
                  policy,
                  batch_size=100,
                  seed=None):

  # Good practice: different seeds for different episodes
  # Seed both the env and the policy
  # Becomes important with parallel simulation
  env_ss, agent_ss = np.random.SeedSequence(seed).spawn(2)
  env_seeds = env_ss.generate_state(batch_size)
  agent_seeds = agent_ss.generate_state(batch_size)

  horizon = env.spec.max_episode_steps
  ds = env.observation_space.shape[0]
  states = torch.zeros(size=(batch_size, horizon, ds)).float()
  actions = torch.zeros(size=(batch_size, horizon)).float()
  rewards = torch.zeros((batch_size, horizon)).float()

  for i in range(batch_size):
    s, _ = env.reset(seed=env_seeds[i])
    policy.seed(seed=agent_seeds[i])

    done = False
    t = 0
    while not done:
      states[i, t, :] = torch.from_numpy(s).float()

      a = policy.act(s)
      actions[i, t] = a

      s, r, terminated, truncated, _ = env.step(a.numpy())
      rewards[i, t,] = r

      done = terminated or truncated
      t += 1

  return states, actions, rewards

Our implementation of REINFORCE will take an env and a policy and will update the policy parameters for a given number (default 200) of iterations. Quite conservatively, we will use a default batch size of 100 and a step size of $10^{-4}$. Feel free to experiment with other values of the hyperparameters.

In [ ]:
def reinforce(env,
              policy,
              batch_size = 100,
              step_size = 1e-4,
              iterations=200,
              seed=None):
  seeds = np.random.SeedSequence(seed).generate_state(iterations)

  performances = []

  for k in range(iterations):
    # Simulation
    ...

    # Gradient estimation
    ...

    # Policy update
    grad = ...

      theta_new = ...

      policy.set_theta(theta_new)

      performance = ...
      print("Iteration %d\nAverage return %f\nParameters %s\nGradient norm %s\n\n"%(
          k,
          performance,
          policy.theta.numpy(),
          torch.linalg.vector_norm(grad).item()))
      performances.append(performance)

  return performances

In [ ]:
env = gym.make("LQR-v0")
pi = LinearGaussianPolicy(sigma=0.1)

performances = reinforce(env, pi, seed=2026)

Let's plot the **learning curve**

In [ ]:
plt.plot(performances)

## 3. Variance Reduction

Try to add an average **baseline** to REINFORCE, and to implement the **GPOMDP** estimator (with and without baseline). Can you observe any improvements in the learning process?

## 4. Multi-Dimensional Actions
Try to solve an LQR with action dimension larger than 1. You will need to make several modifications to the implementation. Most importantly, the policy will be a **multivariate Gaussian**.